# 01 — Scaling Plots

Load results from `results/scaling_v1/results.jsonl` and produce:
- Var(∂θ_i L) vs n per (family, kernel) pair
- Kernel comparison for fixed connectivity
- Init comparison (uniform vs small-angle)
- Scaling law fits (exponential vs polynomial)

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='colorblind')
RESULTS_DIR = Path('../results/scaling_v1')
FIGURES_DIR = Path('../results/scaling_v1/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print('Ready')

## Load results

In [ ]:
records = []
results_file = RESULTS_DIR / 'results.jsonl'
if results_file.exists():
    with open(results_file) as f:
        for line in f:
            records.append(json.loads(line))
    df = pd.DataFrame(records)
    print(f'Loaded {len(df)} records')
    print(df.head())
else:
    print(f'No results yet at {results_file}')
    print('Run: python -m iqp_bp.cli run-scaling configs/experiments/scaling_v1.yaml')
    df = pd.DataFrame()

## Var(∂L) vs n — per (family, kernel)

In [ ]:
if len(df) > 0:
    g = df.groupby(['family', 'kernel', 'init', 'n'])['var'].mean().reset_index()

    for family in g['family'].unique():
        fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=False)
        for ax, kernel in zip(axes, ['gaussian', 'laplacian', 'polynomial', 'linear']):
            sub = g[(g['family'] == family) & (g['kernel'] == kernel)]
            for init in sub['init'].unique():
                row = sub[sub['init'] == init]
                if len(row) > 0:
                    ax.semilogy(row['n'], row['var'], marker='o', label=init)
            ax.set_title(f'{kernel}')
            ax.set_xlabel('n (qubits)')
            ax.set_ylabel('Var(∂L)')
            ax.legend(fontsize=8)
        fig.suptitle(f'Family: {family}', fontsize=13)
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / f'variance_vs_n_{family}.pdf', bbox_inches='tight')
        plt.show()
else:
    print('No data to plot — run experiments first.')

## Scaling law fit: log Var ~ -α·n + β·log(n) + c

In [ ]:
if len(df) > 0:
    from numpy.polynomial import polynomial as P

    fit_records = []
    g2 = df.groupby(['family', 'kernel', 'init', 'n'])['var'].mean().reset_index()

    for (family, kernel, init), sub in g2.groupby(['family', 'kernel', 'init']):
        sub = sub.sort_values('n')
        if len(sub) < 3:
            continue
        ns = sub['n'].values.astype(float)
        log_var = np.log(sub['var'].values.clip(1e-15))
        # Fit: log_var ~ alpha * n + beta * log(n) + c
        X = np.column_stack([ns, np.log(ns), np.ones_like(ns)])
        coeffs, res, _, _ = np.linalg.lstsq(X, log_var, rcond=None)
        alpha, beta, c = coeffs
        fit_records.append({
            'family': family, 'kernel': kernel, 'init': init,
            'alpha': alpha, 'beta': beta, 'c': c,
            'verdict': 'BP' if alpha < -0.05 else ('poly' if beta < -0.3 else 'flat')
        })

    fit_df = pd.DataFrame(fit_records)
    print(fit_df[['family','kernel','init','alpha','beta','verdict']].to_string())
else:
    print('No data.')